In [1]:
import os
os.chdir('/home/ec2-user/SageMaker')
os.system('git clone https://github.com/YinzhiZZZ/MLBA_credit-default-prediction')
os.system('aws s3 cp s3://credit-default-model-mlba/models/xgboost_model.pkl MLBA_credit-default-prediction/models/xgboost_model.pkl')
print("Setup complete!")

fatal: destination path 'MLBA_credit-default-prediction' already exists and is not an empty directory.


download: s3://credit-default-model-mlba/models/xgboost_model.pkl to MLBA_credit-default-prediction/models/xgboost_model.pkl
Setup complete!


In [2]:
import subprocess
result = subprocess.run(['conda', 'install', '-y', '-c', 'conda-forge', 'xgboost'], capture_output=True, text=True)
print("xgboost installed!")

xgboost installed!


In [8]:
import joblib, numpy as np, ipywidgets as widgets, boto3, io
import pandas as pd
from IPython.display import display, clear_output
from datetime import datetime

model = joblib.load('MLBA_credit-default-prediction/models/xgboost_model.pkl')
dynamodb = boto3.resource('dynamodb', region_name='us-east-1')
table = dynamodb.Table('credit-predictions')

style = {'description_width': 'initial'}
layout = widgets.Layout(width='500px')

# ── Tab 1: Manual Input ───────────────────────────────────────────────────────
s_limit = widgets.IntSlider(value=50000, min=10000, max=1000000, step=10000, description='Credit Limit (NT$):', style=style, layout=layout)
s_age   = widgets.IntSlider(value=30, min=18, max=75, description='Age:', style=style, layout=layout)
s_sex   = widgets.Dropdown(options=[('Female',2),('Male',1)], description='Gender:', style=style)
s_edu   = widgets.Dropdown(options=[('University',2),('Graduate School',1),('High School',3),('Other',4)], description='Education:', style=style)
s_mar   = widgets.Dropdown(options=[('Single',2),('Married',1),('Other',3)], description='Marital Status:', style=style)
s_pay0  = widgets.IntSlider(value=-1, min=-2, max=8, description='Pay Status Sep:', style=style, layout=layout)
s_pay2  = widgets.IntSlider(value=-1, min=-2, max=8, description='Pay Status Aug:', style=style, layout=layout)
s_pay3  = widgets.IntSlider(value=-1, min=-2, max=8, description='Pay Status Jul:', style=style, layout=layout)
s_bill1 = widgets.IntText(value=3913, description='Bill Amount Sep (NT$):', style=style)
s_bill2 = widgets.IntText(value=3102, description='Bill Amount Aug (NT$):', style=style)
s_amt1  = widgets.IntText(value=0,    description='Payment Sep (NT$):', style=style)
s_amt2  = widgets.IntText(value=689,  description='Payment Aug (NT$):', style=style)
btn1    = widgets.Button(description='Predict Default Risk', button_style='danger', layout=widgets.Layout(width='300px'))
out1    = widgets.Output()

def go_manual(b):
    out1.clear_output()
    with out1:
        X = np.array([[s_limit.value, s_sex.value, s_edu.value, s_mar.value, s_age.value,
                       s_pay0.value, s_pay2.value, s_pay3.value, -1, -1, -1,
                       s_bill1.value, s_bill2.value, 0, 0, 0, 0,
                       s_amt1.value, s_amt2.value, 0, 0, 0, 0]])
        prob = float(model.predict_proba(X)[0][1])
        risk = "HIGH RISK" if prob > 0.5 else "LOW RISK"
        try:
            table.put_item(Item={
                'id': str(datetime.now().timestamp()),
                'timestamp': str(datetime.now()),
                'source': 'manual',
                'credit_limit': str(s_limit.value),
                'age': str(s_age.value),
                'pay_status_sep': str(s_pay0.value),
                'bill_amt_sep': str(s_bill1.value),
                'pay_amt_sep': str(s_amt1.value),
                'default_probability': str(round(prob, 4)),
                'risk_level': risk
            })
            saved = "Saved to DynamoDB"
        except Exception as e:
            saved = f"DB error: {e}"
        print("⚠️  HIGH RISK" if prob > 0.5 else "✅  LOW RISK")
        print(f"Default Probability: {prob:.1%}")
        print(f"[{saved}]")

btn1.on_click(go_manual)

tab1_content = widgets.VBox([
    widgets.HTML("<h3>Customer Profile</h3>"),
    s_limit, s_age, s_sex, s_edu, s_mar,
    widgets.HTML("<h3>Payment Status (last 3 months)</h3><p>-2: No use | -1: Paid in full | 1~8: Months delayed</p>"),
    s_pay0, s_pay2, s_pay3,
    widgets.HTML("<h3>Bill & Payment Amounts</h3>"),
    s_bill1, s_bill2, s_amt1, s_amt2,
    btn1, out1
])

# ── Tab 2: Batch Prediction from S3 ──────────────────────────────────────────
btn2 = widgets.Button(description='Run Batch Prediction', button_style='warning', layout=widgets.Layout(width='300px'))
out2 = widgets.Output()

def go_batch(b):
    out2.clear_output()
    with out2:
        try:
            df = pd.read_csv('s3://credit-default-model-mlba/data/test_batch.csv')
            print(f"Loaded {len(df)} records from S3")
        except Exception as e:
            print(f"Error reading from S3: {e}")
            return

        feature_cols = ['LIMIT_BAL','SEX','EDUCATION','MARRIAGE','AGE',
                        'PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6',
                        'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6',
                        'PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

        X = df[feature_cols].values
        probs = model.predict_proba(X)[:, 1]
        risks = ["HIGH RISK" if p > 0.5 else "LOW RISK" for p in probs]

        df['default_probability'] = np.round(probs, 4)
        df['risk_level'] = risks

        saved_count = 0
        for i, row in df.iterrows():
            try:
                table.put_item(Item={
                    'id': f"batch_{datetime.now().timestamp()}_{i}",
                    'timestamp': str(datetime.now()),
                    'source': 'batch_s3',
                    'credit_limit': str(int(row['LIMIT_BAL'])),
                    'age': str(int(row['AGE'])),
                    'pay_status_sep': str(int(row['PAY_0'])),
                    'bill_amt_sep': str(int(row['BILL_AMT1'])),
                    'pay_amt_sep': str(int(row['PAY_AMT1'])),
                    'default_probability': str(round(float(row['default_probability']), 4)),
                    'risk_level': row['risk_level']
                })
                saved_count += 1
            except:
                pass

        high_risk = sum(1 for r in risks if r == "HIGH RISK")
        low_risk  = sum(1 for r in risks if r == "LOW RISK")
        print(f"\n=== Batch Prediction Results ===")
        print(f"Data source:       s3://credit-default-model-mlba/data/test_batch.csv")
        print(f"Total records:     {len(df)}")
        print(f"HIGH RISK:         {high_risk} ({high_risk/len(df):.1%})")
        print(f"LOW RISK:          {low_risk} ({low_risk/len(df):.1%})")
        print(f"Saved to DynamoDB: {saved_count} records")
        print(f"\nTop 5 highest risk customers:")
        print(df[['LIMIT_BAL','AGE','default_probability','risk_level']].sort_values('default_probability', ascending=False).head(5).to_string())

btn2.on_click(go_batch)

tab2_content = widgets.VBox([
    widgets.HTML("<h3>Batch Prediction</h3>"),
    widgets.HTML("<p>Reads customer data directly from <b>S3</b> and runs batch predictions. Results are saved to DynamoDB.</p>"),
    btn2,
    out2
])

# ── Tabs ──────────────────────────────────────────────────────────────────────
tab = widgets.Tab()
tab.children = [tab1_content, tab2_content]
tab.set_title(0, 'Single Customer')
tab.set_title(1, 'Batch Prediction (S3)')

display(widgets.HTML("<h2>Credit Card Default Risk Predictor</h2>"), tab)

HTML(value='<h2>Credit Card Default Risk Predictor</h2>')